# Per-rule confidence intervals across five inference methods

This notebook quantifies the uncertainty of every action rule mined from four
real-world tabular datasets, using each of the five confidence-interval (CI)
methods implemented in `action-rules`:

1. **Bootstrap (percentile)** — non-parametric resampling.
2. **Bootstrap (BCa)** — bias-corrected and accelerated (jackknife-based).
3. **Analytic Wald** — closed-form delta-method interval.
4. **Analytic Wilson** — score interval, robust for sparse rules.
5. **Bayesian Beta-Binomial** — Monte Carlo posterior sampling.

For every (dataset × rule × method) triple we record the point estimate, the
95 % interval bounds, and the standard error for both the *uplift* and the
*realistic rule gain* (utility-weighted profit).

**Datasets**: Telco Customer Churn, UCI Bank Marketing, IBM Employee
Attrition, Taiwan Credit Card Default. All four are loaded with the canonical
preprocessing in `notebooks/article/_datasets.py` so results are reproducible.

**Inputs**: CSVs under `notebooks/data/` and `notebooks/ci/data/`.

**Outputs**:
- `notebooks/inference_studies/results/rule_level_cis.csv` — one row per
  (dataset × rule × method).
- `notebooks/inference_studies/results/rule_level_bootstrap_samples.npz` —
  raw bootstrap arrays for the rule with the widest uplift CI on Employee
  Attrition; consumed by the visual diagnostics notebook.

**Runtime**: ~2 minutes on a modern laptop. Deterministic via `random_state=42`.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repository root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

# Import the shared dataset loader (canonical preprocessing for the four
# benchmark datasets used throughout this folder).
sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))
from _datasets import load_all, load_telco, load_bank_marketing, load_employee_attrition, load_credit_card_default  # noqa: E402

# Outputs for this notebook (kept under the notebook folder so the user can
# inspect them locally without touching the article's canonical CSVs).
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'inference_studies' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root:   {REPO_ROOT}')
print(f'Results dir: {RESULTS_DIR.relative_to(REPO_ROOT)}')


In [ ]:
from action_rules import ActionRules
from action_rules.inference.base import extract_rule_masks
from action_rules.inference.bootstrap import _precompute_rule_masks

OUT_CSV = RESULTS_DIR / 'rule_level_cis.csv'
OUT_NPZ = RESULTS_DIR / 'rule_level_bootstrap_samples.npz'


## Parameters

`N_BOOTSTRAP = 500` and `N_MC = 10000` match the article-grade defaults. Drop
them to `100` / `2000` for a quick smoke run.


In [ ]:
N_BOOTSTRAP = 500
N_MC = 10000
SEED = 42
CONFIDENCE_LEVEL = 0.95

METHOD_SPECS = [
    ('bootstrap_percentile', dict(method='bootstrap', bootstrap_type='percentile')),
    ('bootstrap_bca',        dict(method='bootstrap', bootstrap_type='bca')),
    ('wald',                 dict(method='analytic', analytic_type='wald')),
    ('wilson',               dict(method='analytic', analytic_type='wilson')),
    ('bayesian',             dict(method='bayesian')),
]


## Mine rules and compute every CI method on every dataset

For each dataset we (1) mine the rules with the canonical mining
hyperparameters, (2) loop over the five CI methods, and (3) collect the
per-rule numbers into a tidy long-form DataFrame.


In [ ]:
rows = []
saved_samples = {}

for cfg in load_all():
    print(f'=== {cfg.name} ({cfg.df.shape[0]:,} rows) ===')
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    ar.fit(
        cfg.df,
        stable_attributes=cfg.stable_attributes,
        flexible_attributes=cfg.flexible_attributes,
        target=cfg.target,
        target_undesired_state=cfg.target_undesired_state,
        target_desired_state=cfg.target_desired_state,
        use_sparse_matrix=cfg.use_sparse_matrix,
    )
    print(f'  mined {len(ar.output.action_rules)} rules')

    for label, kw in METHOD_SPECS:
        call_kw = dict(kw, confidence_level=CONFIDENCE_LEVEL, random_state=SEED)
        if kw['method'] == 'bootstrap':
            call_kw['n_bootstrap'] = N_BOOTSTRAP
        elif kw['method'] == 'bayesian':
            call_kw['n_mc'] = N_MC
        results = ar.confidence_intervals(cfg.df, **call_kw)
        for r in results:
            rows.append({
                'dataset': cfg.name,
                'rule_index': r.rule_index,
                'method': label,
                'support': r.support,
                'confidence': r.confidence,
                'uplift_point': r.uplift_point,
                'uplift_ci_lower': r.uplift_ci_lower,
                'uplift_ci_upper': r.uplift_ci_upper,
                'uplift_se': r.uplift_se,
                'gain_point': r.realistic_rule_gain_point,
                'gain_ci_lower': r.realistic_rule_gain_ci_lower,
                'gain_ci_upper': r.realistic_rule_gain_ci_upper,
                'gain_se': r.realistic_rule_gain_se,
            })

        # Capture the bootstrap samples and binary masks of the widest-CI
        # rule on Employee Attrition — used by the visual diagnostics notebook
        # to reconstruct the engine's nonparametric bootstrap.
        if label == 'bootstrap_percentile' and cfg.name == 'IBM Employee Attrition':
            ordered = sorted(
                results, key=lambda r: -(r.uplift_ci_upper - r.uplift_ci_lower)
            )
            chosen = ordered[0]
            if chosen.samples_uplift is not None:
                rule_masks = extract_rule_masks(ar.output)
                rm = rule_masks[chosen.rule_index]
                u_ante, u_match, d_ante, d_match = _precompute_rule_masks(cfg.df, rm)
                saved_samples['attrition_rule_uplift'] = chosen.samples_uplift
                saved_samples['attrition_rule_gain'] = (
                    chosen.samples_gain if chosen.samples_gain is not None else np.zeros(0)
                )
                saved_samples['attrition_rule_point'] = np.array([chosen.uplift_point])
                saved_samples['attrition_rule_ci'] = np.array(
                    [chosen.uplift_ci_lower, chosen.uplift_ci_upper]
                )
                saved_samples['attrition_rule_index'] = np.array([chosen.rule_index])
                saved_samples['attrition_rule_u_ante'] = u_ante.astype(np.uint8)
                saved_samples['attrition_rule_u_match'] = u_match.astype(np.uint8)
                saved_samples['attrition_rule_d_ante'] = d_ante.astype(np.uint8)
                saved_samples['attrition_rule_d_match'] = d_match.astype(np.uint8)
                saved_samples['attrition_rule_N'] = np.array([len(cfg.df)])

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
np.savez_compressed(OUT_NPZ, **saved_samples)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)} ({df.shape[0]:,} rows)')
print(f'wrote {OUT_NPZ.relative_to(REPO_ROOT)}')


## Quick inspection: median CI width per method, per dataset

Narrower intervals indicate that the method's variance estimate is smaller,
but only methods with *correct* coverage are trustworthy (see the next
notebook for coverage). For our datasets, bootstrap percentile CIs are
typically a touch wider than the analytic and Bayesian ones, reflecting the
extra Monte Carlo variability.


In [ ]:
df['uplift_width'] = df['uplift_ci_upper'] - df['uplift_ci_lower']
pivot = (
    df.groupby(['dataset', 'method'])['uplift_width']
      .median()
      .unstack('method')
      .round(5)
)
print('Median uplift CI width (smaller = tighter):')
print(pivot.to_string())


## What's in the saved CSV?

`rule_level_cis.csv` is the canonical input to every downstream notebook in
this folder. Columns:

| column | meaning |
| --- | --- |
| `dataset`, `rule_index`, `method` | identifier triple |
| `support`, `confidence` | rule support and confidence on the undesired part |
| `uplift_point`, `uplift_ci_lower`, `uplift_ci_upper`, `uplift_se` | uplift estimates |
| `gain_point`, `gain_ci_lower`, `gain_ci_upper`, `gain_se` | realistic rule gain estimates |

Re-run the notebook end-to-end to refresh the CSV; the random seed is fixed so
results are byte-stable across machines.
